# Quality Suite Smoke Notebook\n\nAssumes API is running on `http://127.0.0.1:8000`.\n\nThis notebook validates quality-focused behavior: realistic calories, diversity controls, and structured infeasibility relaxations.

In [ ]:
import json\nfrom datetime import datetime, timezone\nfrom urllib.request import Request, urlopen\nfrom urllib.error import HTTPError\n\nBASE_URL = \"http://127.0.0.1:8000\"\n\ndef api(method, path, payload=None):\n    url = f\"{BASE_URL}{path}\"\n    body = None\n    headers = {\"Accept\": \"application/json\"}\n    if payload is not None:\n        body = json.dumps(payload).encode(\"utf-8\")\n        headers[\"Content-Type\"] = \"application/json\"\n    req = Request(url, data=body, headers=headers, method=method)\n    try:\n        with urlopen(req, timeout=25) as resp:\n            raw = resp.read().decode(\"utf-8\")\n            return resp.status, json.loads(raw) if raw else {}\n    except HTTPError as exc:\n        raw = exc.read().decode(\"utf-8\")\n        data = json.loads(raw) if raw else {}\n        return exc.code, data\n\ndef show(title, payload):\n    print(f\"\\n=== {title} ===\")\n    print(json.dumps(payload, indent=2, ensure_ascii=False))

In [ ]:
# 1) Create user + profile (no explicit config set)\nstatus, user = api(\"POST\", \"/v1/users\", {\"name\": \"quality-suite\"})\nassert status == 200, (status, user)\nuser_id = user[\"user_id\"]\n\nstatus, _ = api(\n    \"POST\",\n    \"/v1/profile\",\n    {\n        \"user_id\": user_id,\n        \"age\": 35,\n        \"sex\": \"male\",\n        \"height_cm\": 195,\n        \"weight_kg\": 109,\n        \"activity_level\": \"moderate\",\n        \"goal\": \"cut\"\n    }\n)\nassert status == 200\nprint(\"user_id\", user_id)

In [ ]:
# 2) Log and optimize; quality_report should be present\nnow = datetime.now(timezone.utc).isoformat()\nstatus, logged = api(\n    \"POST\",\n    \"/v1/logs/select\",\n    {\"user_id\": user_id, \"timestamp\": now, \"food_id\": \"simit\", \"grams\": 120}\n)\nassert status == 200, (status, logged)\n\nstatus, optimized = api(\"POST\", \"/v1/plan/optimize\", {\"user_id\": user_id})\nassert status == 200, (status, optimized)\nassert \"quality_report\" in optimized\nshow(\"optimize quality_report\", optimized[\"quality_report\"])

In [ ]:
# 3) Tight diversity / share settings and verify response still structured\nstatus, cfg = api(\n    \"PUT\",\n    \"/v1/config\",\n    {\n      \"user_id\": user_id,\n      \"config\": {\n        \"horizon_days\": 1,\n        \"constraints\": {\n          \"calories_kcal\": {\"min\": 1800, \"max\": 2600},\n          \"protein_g\": {\"min\": 120, \"max\": 220}\n        },\n        \"objectives_lex\": [\n          {\n            \"name\": \"min_dev\",\n            \"type\": \"deviation\",\n            \"targets\": [\"calories_kcal\", \"protein_g\"],\n            \"target_values\": {\"calories_kcal\": 2200, \"protein_g\": 170}\n          },\n          {\"name\": \"min_cost\", \"type\": \"linear\", \"metric\": \"cost_try\", \"sense\": \"min\"}\n        ],\n        \"diversity\": {\n          \"enabled\": true,\n          \"max_single_food_calorie_share\": 0.50,\n          \"min_variety_count\": 3,\n          \"variety_min_grams\": 20\n        }\n      }\n    }\n)\nassert status == 200, (status, cfg)\nshow(\"put config warnings\", cfg)

In [ ]:
# 4) Force infeasibility and inspect suggested_relaxations (expect 422)\nstatus, cfg = api(\n    \"PUT\",\n    \"/v1/config\",\n    {\n      \"user_id\": user_id,\n      \"config\": {\n        \"horizon_days\": 1,\n        \"constraints\": {\n          \"calories_kcal\": {\"max\": 300},\n          \"protein_g\": {\"min\": 220}\n        },\n        \"objectives_lex\": [\n          {\"name\": \"min_dev\", \"type\": \"deviation\", \"targets\": [\"calories_kcal\", \"protein_g\"]}\n        ],\n        \"diversity\": {\n          \"enabled\": true,\n          \"max_single_food_calorie_share\": 0.2,\n          \"min_variety_count\": 5,\n          \"variety_min_grams\": 80\n        },\n        \"food_bounds\": {\"min_grams_per_food\": 0, \"max_grams_per_food\": 60}\n      }\n    }\n)\nassert status == 200\n\nstatus, infeasible = api(\"POST\", \"/v1/plan/optimize\", {\"user_id\": user_id})\nassert status == 422, (status, infeasible)\nshow(\"infeasible relaxations\", infeasible)